In [2]:
import torch
import torch.nn as nn
# from utils import  intersection


In [ ]:
class YoloLoss(nn.Module):
    def __init__(self,S=7,B=2,c=20):
        super(YoloLoss,self).__init__()
        self.mse=nn.MSELoss(reduction="sum")
        self.S=S 
        self.B=B 
        self.c=C
        self.lambda_noobj=0.5
        self.lambda_coord=5

    def forward(self,predictions,target):
        predictions=predictions.reshape(-1,self.S,self.S,self.C+self.B*5)

        iou_b1=intersection_over_union(predictions[...,21:25],target[...,21:25])
        iou_b2=intersection_over_union(predictions[...,26:30],target[...,26:30])
        ious=torch.cat([iou_b1.unsqueeze(0),iou_b2.unsqueeze(0)],dim=0)
        exists_box = target[..., 20].unsqueeze(3)  #identity of the object i  (i there is any object in cell i)

        # bbox cordinates

        box_predictions=exists_box*(
            (
            best_box*predictions[...,21:25]
            +(1-bestbox)* predictions[...,21:25]
            )
        )

        box_targets=exists_box*target[...,21:25]
        box_predictions[...,2:4]=torch.sign(box_predictions[...,2:4]) * torch.sqrt(torch.abs(box_predictions[...,2:4]+1e-6))

        box_targets[...,2:4]=torch.sqrt(boc_targets[...,2:4])

        # (N,S,S,25) -> (N*S*S,4)

        box_loss=self.mse(
            torch.flatten(box_predictions,),
            torch.flatten(box_targets,end_dim=-2),
        )

        # object loss

        pred_box=(
            best_box*predictions[...,21:25] + (1-bestbox)* predictions[...,21:25]

        )

        # (N*S*S,1)
        object_loss=self.mse(
            torch.flatten(exists_box * pred_box),
            torch.flatten(exists_box*target[...,20:21])
        )

        #loss for no object 

        no_object=self.mse(
            torch.flatten((1-exists_box)*predictions[...,20:21],start_dim=1),
            torch.flatten((1-exists_box)*target[...,20:21],start_dim=1),
        )

        no_object_loss +=self.mse(
            torch.flatten((1-exists_box)*predictions[...,20:21],start_dims=1),
            torch.flatten((1-exists_box)*target[...,20:21],start_dim=1),
        )

        #for class lsoss 

        obj=self.mse(
            torch.flatten((1-exists_box)*predictions[])
        )




